In [1]:
import json
from pathlib import Path
import xml.etree.ElementTree as ET


# ============================================================
# 1. ORDNER
# ============================================================

INPUT_DIR = Path("json_with_positions")
OUTPUT_DIR = Path("tei")
ERROR_FILE = Path("tei_errors.txt")

OUTPUT_DIR.mkdir(exist_ok=True)


# ============================================================
# 2. XML-NAMESPACE
# ============================================================

TEI_NS = "http://www.tei-c.org/ns/1.0"

ET.register_namespace("", TEI_NS)


# ============================================================
# 3. TEI-ELEMENT ERSTELLEN
# ============================================================

def create_tei_element(annotation, text_content):

    element_name = annotation["element"]
    attributes = annotation.get("attributes", {})

    element = ET.Element(
        f"{{{TEI_NS}}}{element_name}"
    )

    # Attribute übertragen
    for key, value in attributes.items():

        if value is not None:
            element.set(key, str(value))

    # Text innerhalb des Elements
    element.text = text_content

    return element


# ============================================================
# 4. ANNOTATIONEN VALIDIEREN
# ============================================================

def validate_annotations(text, annotations):

    errors = []

    for i, annotation in enumerate(annotations):

        if "start" not in annotation or "end" not in annotation:

            errors.append(
                f'Annotation {i} "{annotation.get("text", "")}" '
                f'hat keine start/end-Position.'
            )

            continue

        start = annotation["start"]
        end = annotation["end"]
        annotation_text = annotation.get("text", "")

        # Position innerhalb des Textes?
        if start < 0 or end > len(text):

            errors.append(
                f'Annotation {i} "{annotation_text}": '
                f'Position außerhalb des Textes ({start}, {end}).'
            )

            continue

        # Stimmen Position und Annotationstext überein?
        actual_text = text[start:end]

        if actual_text != annotation_text:

            errors.append(
                f'Annotation {i} "{annotation_text}": '
                f'Position {start}:{end} ergibt "{actual_text}".'
            )

    return errors


# ============================================================
# 5. TEI-INHALT AUS ANNOTATIONEN ERZEUGEN
# ============================================================

def build_tei_paragraph(text, annotations):

    # --------------------------------------------------------
    # Annotationen mit gültigen Positionen
    # --------------------------------------------------------

    valid_annotations = [
        a for a in annotations
        if "start" in a and "end" in a
    ]

    # --------------------------------------------------------
    # Nach Startposition sortieren
    # Bei gleicher Startposition:
    # längere Annotation zuerst
    #
    # Dadurch wird z.B.
    #
    # person:      9-21
    # occupation:  9-21
    #
    # zusammen behandelt.
    # --------------------------------------------------------

    valid_annotations.sort(
        key=lambda a: (
            a["start"],
            -(a["end"] - a["start"])
        )
    )

    # --------------------------------------------------------
    # Wir bauen das Paragraph-Element
    # --------------------------------------------------------

    p = ET.Element(f"{{{TEI_NS}}}p")

    current_position = 0

    i = 0

    while i < len(valid_annotations):

        annotation = valid_annotations[i]

        start = annotation["start"]
        end = annotation["end"]

        # ----------------------------------------------------
        # Text vor der Annotation
        # ----------------------------------------------------

        if start > current_position:

            plain_text = text[current_position:start]

            if len(p) == 0:
                p.text = (p.text or "") + plain_text
            else:
                previous = p[-1]
                previous.tail = (previous.tail or "") + plain_text

        # ----------------------------------------------------
        # Prüfen, ob mehrere Annotationen exakt dieselbe
        # Textstelle annotieren.
        #
        # Beispiel:
        #
        # person       9-21
        # occupation   9-21
        # ----------------------------------------------------

        same_position = []

        j = i

        while j < len(valid_annotations):

            candidate = valid_annotations[j]

            if (
                candidate["start"] == start
                and candidate["end"] == end
            ):
                same_position.append(candidate)
                j += 1

            else:
                break

        # ----------------------------------------------------
        # Verschachtelte Annotationen
        #
        # Aktuell speziell für:
        #
        # person
        # └── occupation
        # ----------------------------------------------------

        if len(same_position) > 1:

            person_annotation = None
            occupation_annotation = None
            other_annotations = []

            for a in same_position:

                if a["element"] == "person":
                    person_annotation = a

                elif a["element"] == "occupation":
                    occupation_annotation = a

                else:
                    other_annotations.append(a)

            if person_annotation and occupation_annotation:

                person_element = create_tei_element(
                    person_annotation,
                    text[start:end]
                )

                occupation_element = create_tei_element(
                    occupation_annotation,
                    text[start:end]
                )

                # occupation innerhalb von person
                person_element.remove(
                    person_element
                ) if False else None

                # Text des person-Elements entfernen,
                # damit occupation darin eingesetzt werden kann
                person_element.text = None

                occupation_element.text = text[start:end]

                person_element.append(
                    occupation_element
                )

                p.append(person_element)

            else:

                # Sicherheitsfall:
                # mehrere Annotationen, aber keine definierte
                # Verschachtelungsregel
                for a in same_position:

                    element = create_tei_element(
                        a,
                        text[start:end]
                    )

                    p.append(element)

                    # Warnung könnte später ergänzt werden

        else:

            # ------------------------------------------------
            # Normale einzelne Annotation
            # ------------------------------------------------

            element = create_tei_element(
                annotation,
                text[start:end]
            )

            p.append(element)

        current_position = end
        i = j

    # --------------------------------------------------------
    # Restlicher Text nach der letzten Annotation
    # --------------------------------------------------------

    if current_position < len(text):

        remaining_text = text[current_position:]

        if len(p) == 0:
            p.text = (p.text or "") + remaining_text
        else:
            previous = p[-1]
            previous.tail = (previous.tail or "") + remaining_text

    return p


# ============================================================
# 6. TEI-DOKUMENT ERSTELLEN
# ============================================================

def create_tei_document(data):

    advertisement_id = data["id"]
    text = data["text"]
    annotations = data.get("annotations", [])

    # --------------------------------------------------------
    # TEI
    # --------------------------------------------------------

    tei = ET.Element(
        f"{{{TEI_NS}}}TEI"
    )

    # --------------------------------------------------------
    # HEADER
    # --------------------------------------------------------

    tei_header = ET.SubElement(
        tei,
        f"{{{TEI_NS}}}teiHeader"
    )

    file_desc = ET.SubElement(
        tei_header,
        f"{{{TEI_NS}}}fileDesc"
    )

    # titleStmt
    title_stmt = ET.SubElement(
        file_desc,
        f"{{{TEI_NS}}}titleStmt"
    )

    title = ET.SubElement(
        title_stmt,
        f"{{{TEI_NS}}}title"
    )

    title.text = f"Stellenanzeigen im Sydsvenska Dagbladet, 1906 {advertisement_id}"

    # publicationStmt
    publication_stmt = ET.SubElement(
        file_desc,
        f"{{{TEI_NS}}}publicationStmt"
    )

    publication_p = ET.SubElement(
        publication_stmt,
        f"{{{TEI_NS}}}p"
    )

    publication_p.text = "Digitale Edition im Rahmen eines Uniprojekts"

    # sourceDesc
    source_desc = ET.SubElement(
        file_desc,
        f"{{{TEI_NS}}}sourceDesc"
    )

    bibl = ET.SubElement(
        source_desc,
        f"{{{TEI_NS}}}bibl"
    )

    newspaper = ET.SubElement(
        bibl,
        f"{{{TEI_NS}}}title"
    )

    newspaper.text = "Sydsvenska Dagbladet"

    # Datum aus ID
    date_string = advertisement_id[:10]

    date = ET.SubElement(
        bibl,
        f"{{{TEI_NS}}}date"
    )

    date.set("when", date_string)
    date.text = date_string

    # --------------------------------------------------------
    # TEXT
    # --------------------------------------------------------

    text_element = ET.SubElement(
        tei,
        f"{{{TEI_NS}}}text"
    )

    body = ET.SubElement(
        text_element,
        f"{{{TEI_NS}}}body"
    )

    div = ET.SubElement(
        body,
        f"{{{TEI_NS}}}div"
    )

    div.set(
        "{http://www.w3.org/XML/1998/namespace}id",
        advertisement_id
    )

    # --------------------------------------------------------
    # Paragraph mit Annotationen
    # --------------------------------------------------------

    paragraph = build_tei_paragraph(
        text,
        annotations
    )

    div.append(paragraph)

    return tei


# ============================================================
# 7. XML SPEICHERN
# ============================================================

def save_tei(tei, output_file):

    tree = ET.ElementTree(tei)

    ET.indent(tree, space="    ")

    tree.write(
        output_file,
        encoding="utf-8",
        xml_declaration=True
    )


# ============================================================
# 8. ALLE DATEIEN VERARBEITEN
# ============================================================

all_errors = []

json_files = list(INPUT_DIR.glob("*.json"))

print(f"{len(json_files)} JSON-Dateien gefunden.\n")


for json_file in json_files:

    try:

        # JSON lesen
        with open(
            json_file,
            "r",
            encoding="utf-8"
        ) as f:

            data = json.load(f)

        text = data["text"]
        annotations = data.get("annotations", [])

        # ----------------------------------------------------
        # Validierung
        # ----------------------------------------------------

        errors = validate_annotations(
            text,
            annotations
        )

        if errors:

            print(f"⚠ {json_file.name}")

            for error in errors:
                print("   ", error)

            all_errors.append(
                (json_file.name, errors)
            )

        # ----------------------------------------------------
        # TEI erzeugen
        # ----------------------------------------------------

        tei = create_tei_document(data)

        output_file = (
            OUTPUT_DIR /
            f"{data['id']}.xml"
        )

        save_tei(
            tei,
            output_file
        )

        print(f"✓ {json_file.name} → {output_file.name}")

    except Exception as e:

        print(f"❌ {json_file.name}: {e}")

        all_errors.append(
            (
                json_file.name,
                [str(e)]
            )
        )


# ============================================================
# 9. FEHLERBERICHT
# ============================================================

with open(
    ERROR_FILE,
    "w",
    encoding="utf-8"
) as f:

    if not all_errors:

        f.write(
            "Keine Fehler gefunden.\n"
        )

    else:

        for filename, errors in all_errors:

            f.write(
                f"{filename}\n"
            )

            f.write(
                "-" * len(filename) + "\n"
            )

            for error in errors:

                f.write(
                    f"- {error}\n"
                )

            f.write("\n")


# ============================================================
# 10. ABSCHLUSS
# ============================================================

print("\n================================")
print("TEI-Umwandlung abgeschlossen.")
print(f"JSON-Dateien: {len(json_files)}")
print(f"Dateien mit Fehlern: {len(all_errors)}")
print(f"TEI-Ordner: {OUTPUT_DIR}")
print(f"Fehlerbericht: {ERROR_FILE}")
print("================================")

446 JSON-Dateien gefunden.

✓ 1906_01_03_SD_PS_001.json → 1906_01_03_SD_PS_001.xml
✓ 1906_01_05_SD_PS_001.json → 1906_01_05_SD_PS_001.xml
✓ 1906_01_07_SD_PS_001.json → 1906_01_07_SD_PS_001.xml
✓ 1906_01_08_SD_PS_001.json → 1906_01_08_SD_PS_001.xml
✓ 1906_01_08_SD_PS_002.json → 1906_01_08_SD_PS_002.xml
✓ 1906_01_09_SD_PS_001.json → 1906_01_09_SD_PS_001.xml
✓ 1906_01_14_SD_PS_001.json → 1906_01_14_SD_PS_001.xml
✓ 1906_01_14_SD_PS_002.json → 1906_01_14_SD_PS_002.xml
✓ 1906_01_14_SD_PS_003.json → 1906_01_14_SD_PS_003.xml
✓ 1906_01_18_SD_LP_001.json → 1906_01_18_SD_LP_001.xml
✓ 1906_01_19_SD_PS_001.json → 1906_01_19_SD_PS_001.xml
✓ 1906_01_20_SD_PS_001.json → 1906_01_20_SD_PS_001.xml
✓ 1906_01_21_SD_LP_001.json → 1906_01_21_SD_LP_001.xml
✓ 1906_01_21_SD_PS_001.json → 1906_01_21_SD_PS_001.xml
✓ 1906_01_22_SD_PS_001.json → 1906_01_22_SD_PS_001.xml
✓ 1906_01_24_SD_PS_001.json → 1906_01_24_SD_PS_001.xml
✓ 1906_01_25_SD_LP_001.json → 1906_01_25_SD_LP_001.xml
✓ 1906_01_26_SD_LP_001.json → 1906_01

✓ 1906_07_10_SD_PS_001.json → 1906_07_10_SD_PS_001.xml
✓ 1906_07_12_SD_LP_001.json → 1906_07_12_SD_LP_001.xml
✓ 1906_07_12_SD_PS_001.json → 1906_07_12_SD_PS_001.xml
✓ 1906_07_14_SD_PS_001.json → 1906_07_14_SD_PS_001.xml
✓ 1906_07_15_SD_LP_001.json → 1906_07_15_SD_LP_001.xml
✓ 1906_07_15_SD_PS_001.json → 1906_07_15_SD_PS_001.xml
✓ 1906_07_16_SD_PS_001.json → 1906_07_16_SD_PS_001.xml
✓ 1906_07_18_SD_LP_001.json → 1906_07_18_SD_LP_001.xml
✓ 1906_07_20_SD_PS_001.json → 1906_07_20_SD_PS_001.xml
✓ 1906_07_21_SD_LP_001.json → 1906_07_21_SD_LP_001.xml
✓ 1906_07_21_SD_PS_001.json → 1906_07_21_SD_PS_001.xml
✓ 1906_07_22_SD_LP_001.json → 1906_07_22_SD_LP_001.xml
✓ 1906_07_22_SD_LP_002.json → 1906_07_22_SD_LP_002.xml
✓ 1906_07_22_SD_PS_001.json → 1906_07_22_SD_PS_001.xml
✓ 1906_07_22_SD_PS_002.json → 1906_07_22_SD_PS_002.xml
✓ 1906_07_23_SD_LP_001.json → 1906_07_23_SD_LP_001.xml
✓ 1906_07_23_SD_LP_002.json → 1906_07_23_SD_LP_002.xml
✓ 1906_07_23_SD_PS_001.json → 1906_07_23_SD_PS_001.xml
✓ 1906_07_

✓ 1906_09_15_SD_LP_001.json → 1906_09_15_SD_LP_001.xml
✓ 1906_09_15_SD_PS_001.json → 1906_09_15_SD_PS_001.xml
✓ 1906_09_16_SD_LP_001.json → 1906_09_16_SD_LP_001.xml
✓ 1906_09_16_SD_LP_002.json → 1906_09_16_SD_LP_002.xml
✓ 1906_09_16_SD_LP_003.json → 1906_09_16_SD_LP_003.xml
✓ 1906_09_16_SD_LP_004.json → 1906_09_16_SD_LP_004.xml
✓ 1906_09_16_SD_PS_001.json → 1906_09_16_SD_PS_001.xml
✓ 1906_09_16_SD_PS_002.json → 1906_09_16_SD_PS_002.xml
✓ 1906_09_16_SD_PS_003.json → 1906_09_16_SD_PS_003.xml
✓ 1906_09_16_SD_PS_004.json → 1906_09_16_SD_PS_004.xml
✓ 1906_09_16_SD_PS_005.json → 1906_09_16_SD_PS_005.xml
✓ 1906_09_17_SD_PS_001.json → 1906_09_17_SD_PS_001.xml
✓ 1906_09_18_SD_LP_001.json → 1906_09_18_SD_LP_001.xml
✓ 1906_09_18_SD_PS_001.json → 1906_09_18_SD_PS_001.xml
✓ 1906_09_19_SD_LP_001.json → 1906_09_19_SD_LP_001.xml
✓ 1906_09_19_SD_LP_002.json → 1906_09_19_SD_LP_002.xml
✓ 1906_09_20_SD_LP_001.json → 1906_09_20_SD_LP_001.xml
✓ 1906_09_21_SD_LP_001.json → 1906_09_21_SD_LP_001.xml
✓ 1906_09_